# Financial Market Regime & Event Intelligence Engine
## Notebook 04: Financial News & Event Data Ingestion Prototype

Welcome to Notebook 04! Having developed quantitative regime detection models (K-Means in Notebook 01 and Gaussian HMM in Notebook 02 & 03), we now build the **Financial News & Event Data Ingestion Pipeline**.

### Objectives:
1. **Document Data Source**: Describe the public financial news API (`yfinance`), endpoint specifications, capabilities, and limitations.
2. **Acquire Market News**: Ingest financial news and macro events relevant to the S&P 500 and U.S. financial markets.
3. **Clean & Standardize Pipeline**: Convert timestamps to UTC datetime, remove duplicate headlines, and handle missing attributes.
4. **Data Inspection**: Display initial records, DataFrame shape, missing-value counts, date range, and publisher metrics.
5. **Timeline Visualization**: Create an interactive Plotly timeline of collected news events.
6. **Architectural Explanation**: Document why news and event intelligence are crucial for interpreting market regime shifts.

---
### Step 1: Public Financial News Source Documentation

#### Selected Source: Yahoo Finance Public News API (`yfinance`)

- **Access Method**: Python `yfinance` library (`yf.Ticker(symbol).news`).
- **Cost & Authentication**: Completely free and public; requires **no paid subscription** or API key.
- **Provided Fields**:
  - `title`: Headline of the news article.
  - `pubDate`: Publication timestamp in ISO 8601 UTC format.
  - `summary` / `description`: Brief text overview or snippet of the article.
  - `provider`: Publisher/source attribution (e.g., Yahoo Finance, Reuters, Bloomberg, MarketWatch).
  - `canonicalUrl` / `clickThroughUrl`: Direct URL link to the original article.
  - `relatedTickers`: Associated financial instruments or entity tags.

#### Limitations:
1. **Recent News Window**: The public endpoint returns the most recent 10–20 news stories per ticker rather than multi-year historical archives.
2. **Rate Limits**: Excessive rapid polling can result in temporary rate-limiting.

#### Credentials & Environment Setup Note:
While `yfinance` requires no API keys, future integration with optional external news providers (e.g., NewsAPI, Finnhub, or Alpha Vantage) will use environment variables (`os.getenv("FINANCIAL_NEWS_API_KEY")`) rather than hardcoding secrets in code.

---
### Step 2: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Data manipulation, timestamp handling, and tabular reporting.
- `plotly.express`: Interactive visualizations of publication timelines.
- `yfinance`: Ingesting live financial news data.
- `os`: Interfacing with environment configurations.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print("Libraries successfully imported!")

Libraries successfully imported!


---
### Step 3: Retrieve Financial News Sample Across Market Tickers

We query news across major market benchmarks, sector ETFs, and top S&P 500 constituents to form a representative sample of U.S. financial market news:
- Index & Macro Benchmarks: `^GSPC` (S&P 500), `SPY` (S&P ETF), `^VIX` (Volatility), `TLT` (Treasuries), `GLD` (Gold), `QQQ` (Nasdaq 100)
- Large-Cap Market Drivers: `AAPL`, `MSFT`, `NVDA`, `AMZN`, `JPM`, `GS`

In [2]:
target_tickers = [
    "^GSPC", "SPY", "^VIX", "TLT", "GLD", "QQQ",
    "AAPL", "MSFT", "NVDA", "AMZN", "JPM", "GS"
]

raw_news_records = []

for ticker in target_tickers:
    try:
        news_list = yf.Ticker(ticker).news
        for item in news_list:
            # yfinance returns dictionary with 'content' sub-dict
            content = item.get("content", item)
            
            headline = content.get("title")
            pub_date = content.get("pubDate")
            summary = content.get("summary") or content.get("description", "")
            
            # Provider info
            provider_info = content.get("provider", {})
            publisher = provider_info.get("displayName", "Unknown") if isinstance(provider_info, dict) else "Unknown"
            
            # URL info
            canonical_info = content.get("canonicalUrl", {})
            click_info = content.get("clickThroughUrl", {})
            url = canonical_info.get("url") if isinstance(canonical_info, dict) and canonical_info.get("url") else (click_info.get("url", "") if isinstance(click_info, dict) else "")
            
            raw_news_records.append({
                "query_ticker": ticker,
                "headline": headline,
                "pub_date_raw": pub_date,
                "summary": summary,
                "publisher": publisher,
                "url": url
            })
    except Exception as e:
        print(f"Warning: Failed to fetch news for {ticker}: {e}")

raw_df = pd.DataFrame(raw_news_records)
print(f"Total raw news records retrieved: {len(raw_df)}")

Total raw news records retrieved: 120


---
### Step 4: Data Cleaning, Timestamp Normalization & Deduplication

1. **Timestamp Conversion**: Parse raw publication dates into a standardized UTC `datetime64[ns, UTC]` format.
2. **Deduplication**: Remove duplicate articles sharing identical headlines.
3. **Missing Value Handling**: Fill null summaries with `"N/A"` and null publishers with `"Unknown"`.

In [3]:
# 1. Convert timestamps to standardized UTC datetime
raw_df["published_at"] = pd.to_datetime(raw_df["pub_date_raw"], utc=True)

# 2. Calculate duplicate count before cleaning
initial_count = len(raw_df)
duplicate_count = raw_df.duplicated(subset=["headline"]).sum()

# 3. Drop missing headlines and duplicate stories
clean_news_df = raw_df.dropna(subset=["headline"]).drop_duplicates(subset=["headline"]).copy()

# 4. Fill missing text attributes
clean_news_df["summary"] = clean_news_df["summary"].fillna("N/A")
clean_news_df["publisher"] = clean_news_df["publisher"].fillna("Unknown")
clean_news_df["url"] = clean_news_df["url"].fillna("")

# Reorder columns cleanly
final_cols = ["published_at", "headline", "publisher", "query_ticker", "summary", "url"]
clean_news_df = clean_news_df[final_cols].sort_values(by="published_at", ascending=False).reset_index(drop=True)

# Display Data Inspection Metrics
print("=== Financial News Dataset Summary ===")
print(f"DataFrame Shape: {clean_news_df.shape}")
print(f"Column Names: {list(clean_news_df.columns)}")
print(f"Total Raw Articles Ingested: {initial_count}")
print(f"Duplicate Headlines Removed: {duplicate_count}")
print(f"Unique Publishers Count: {clean_news_df['publisher'].nunique()}")
print(f"Date Range: {clean_news_df['published_at'].min()}  to  {clean_news_df['published_at'].max()}\n")

print("--- Missing Values Count ---")
display(clean_news_df.isnull().sum())

print("\n--- First 10 News Records ---")
display(clean_news_df.head(10))

=== Financial News Dataset Summary ===
DataFrame Shape: (108, 6)
Column Names: ['published_at', 'headline', 'publisher', 'query_ticker', 'summary', 'url']
Total Raw Articles Ingested: 120
Duplicate Headlines Removed: 12
Unique Publishers Count: 29
Date Range: 2026-09-08 20:54:00+00:00  to  2026-09-19 15:11:11+00:00

--- Missing Values Count ---


published_at    0
headline        0
publisher       0
query_ticker    0
summary         0
url             0
dtype: int64


--- First 10 News Records ---


,published_at,headline,publisher,query_ticker,summary,url
0,2026-09-19 15:11:11+00:00,"Coinbase Files With CFTC To List US Single-Stock Perpetual Futures, Includin...",Stocktwits,AAPL,"Coinbase’s filing used an Apple contract as its model, which is cash-settled...",https://stocktwits.com/news-articles/markets/equity/coinbase-apple-tesla-per...
1,2026-09-19 15:05:00+00:00,The Stock Market's Biggest Companies Are Losing Their Grip. Here's the ETF I...,Motley Fool,^GSPC,"As the market rotates away from large-cap stocks, new opportunities have eme...",https://www.fool.com/investing/2026/09/19/stock-market-biggest-companies-los...
2,2026-09-19 14:50:00+00:00,"Warren Buffett Watched Alphabet's Stock Price Climb 9,000% Before He Decided...",Motley Fool,NVDA,Here's why Berkshire could be buying more Alphabet shares this quarter.,https://www.fool.com/investing/2026/09/19/warren-buffett-watched-alphabets-s...
3,2026-09-19 14:33:06+00:00,CVS division completes Chapter 11 bankruptcy liquidation,TheStreet,AMZN,A piece of the company is winding down its operations under CVS Health.,https://www.thestreet.com/retail/cvs-omnicare-completes-chapter-11-bankruptc...
4,2026-09-19 14:32:00+00:00,"Upstart, Affirm, and SoFi All Lend to the Same Borrowers. Only One of Them F...",Motley Fool,NVDA,There has been an increase in the number of fintechs applying for bank chart...,https://www.fool.com/investing/2026/09/19/upstart-affirm-and-sofi-all-lend-t...
5,2026-09-19 14:10:33+00:00,How Investors Are Reacting To Ultragenyx Pharmaceutical (RARE) First-In-Dise...,Simply Wall St.,MSFT,"In September 2026, Ultragenyx Pharmaceutical received full U.S. FDA approval...",https://finance.yahoo.com/healthcare/articles/investors-reacting-ultragenyx-...
6,2026-09-19 14:05:00+00:00,Anthropic's IPO Is Coming. Here's What That Means for S&P 500 Investors.,Motley Fool,^GSPC,"The market will be highly affected by this IPO, even if the stock isn't incl...",https://www.fool.com/investing/2026/09/19/anthropic-ipo-coming-what-means-sp...
7,2026-09-19 14:02:51+00:00,IBM Has Raised Its Dividend for 31 Years. Inflation Is Still Winning,24/7 Wall St.,MSFT,"IBM has raised its dividend for three decades straight, but the latest incre...",https://247wallst.com/investing/2026/09/19/ibm-has-raised-its-dividend-for-3...
8,2026-09-19 13:50:00+00:00,Has AGNC's Monthly Dividend Made Up for What Its Share Price Did?,Motley Fool,NVDA,AGNC Investment's dividend payments have really added up over the years.,https://www.fool.com/investing/2026/09/19/has-agncs-monthly-dividend-made-up...
9,2026-09-19 13:42:00+00:00,"Bank of America Sets 12-Month S&P 500 Target at 7,800, Sees 2% Upside",InvestorsHub,^GSPC,"Bank of America (BofA) has introduced a 12-month target of 7,800 for the S&P...",https://investorshub.advfn.com/market-news/article/36524/bank-of-america-set...


---
### Step 5: Visualize Collected News Events Timeline

We visualize the temporal distribution of collected news stories over time.

In [4]:
# Group articles by publication date (daily count)
timeline_df = clean_news_df.copy()
timeline_df["date_only"] = timeline_df["published_at"].dt.date
daily_counts = timeline_df.groupby("date_only").size().reset_index(name="article_count")

fig_news_ts = px.bar(
    daily_counts,
    x="date_only",
    y="article_count",
    title="Collected Financial News Articles Volume by Date",
    labels={"date_only": "Publication Date", "article_count": "Number of Articles"},
    template="plotly_white"
)

fig_news_ts.update_layout(title_x=0.5)
fig_news_ts.show()

---
### Step 6: Architectural Role of Event Intelligence in Market Regime Detection

#### Why News & Event Data Matter for the Engine:
1. **Catalysts Behind Regime Transitions**:
   - Quantitative market features (rolling volatility, momentum, drawdown) inform us **THAT** a market regime transition occurred (e.g., moving from low volatility to a high volatility crisis).
   - Macroeconomic news, central bank announcements (Federal Reserve rate decisions), inflation CPI releases, bank failures, and geopolitical events represent the underlying **catalysts** that trigger these transitions.

2. **Contextual Event Intelligence**:
   - Ingesting structured financial news allows our engine to link quantitative regime changes detected by K-Means or HMM with human-readable news context, providing actionable intelligence for portfolio managers and risk systems.

--- 
### Step 7: Next Pipeline Steps (Future Scope)
- In future modules, acquired news text will undergo Natural Language Processing (NLP), sentiment extraction (FinBERT), and event classification before being integrated with the quantitative regime engine.